In [ ]:
!pip install evaluate bert_score sentence-transformers faiss-gpu-cu12

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import os
from tqdm import tqdm
import torch
import re
from evaluate import load
import json
from sentence_transformers import SentenceTransformer, util
from datasets import Dataset
import faiss
import traceback
import numpy as np

In [ ]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        trust_remote_code=True
    )

    return tokenizer, model

In [ ]:
model_name = "google/gemma-3-1b-it"

tokenizer, model = load_model(model_name)

Prepare the dataset

In [ ]:
def clean_punctuation(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r"\s'(\w)", r"'\1", text)
    text = re.sub(r"\s([.,!?;:])", r"\1", text)
    text = text.replace(" n't", "n't")
    return text.strip()

In [ ]:
df = pd.read_csv("ru_idioms_corpus.csv")

text_columns = ['Literal_Sent', 'Idiomatic_Sent', 'Idiom']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_punctuation)

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df, groups=df['Idiom']))

train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['Idiom']))

train_df = train_val_df.iloc[train_idx]
val_df = train_val_df.iloc[val_idx]

print(f"Idioms in Train: {train_df['Idiom'].nunique()}, Lines: {len(train_df)}")
print(f"Idioms in Val: {val_df['Idiom'].nunique()}, Lines: {len(val_df)}")
print(f"Idioms in Test: {test_df['Idiom'].nunique()}, Lines: {len(test_df)}")

kNN-LM

In [ ]:
def calculate_metrics(csv_path, ref_lookup, lang="en"):
    df = pd.read_csv(csv_path)

    meteor_metric = load("meteor")
    bertscore_metric = load("bertscore")
    bert_score_model = "roberta-large" if lang == "en" else "xlm-roberta-large"

    if lang == "en":
        sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    else:
        sbert_model = SentenceTransformer('sentence-transformers/LaBSE')


    preds = df['generated_text'].fillna("").tolist()
    inputs = df['input_text'].tolist()

    refs = [ref_lookup.get(inp, [df.iloc[i]['reference']]) for i, inp in enumerate(inputs)]

    meteor_res = meteor_metric.compute(predictions=preds, references=refs)

    all_bs_f1 = []
    for p, rs in zip(preds, refs):
        res = bertscore_metric.compute(predictions=[p] * len(rs), references=rs, lang=lang, model_type=bert_score_model)
        all_bs_f1.append(max(res['f1']))

    max_cosines = []
    for pred, rs in zip(preds, refs):
        p_emb = sbert_model.encode([pred], convert_to_tensor=True)
        r_embs = sbert_model.encode(rs, convert_to_tensor=True)
        scores = util.cos_sim(p_emb, r_embs)
        max_cosines.append(scores.max().item())

    return {
        "METEOR": round(meteor_res['meteor'], 4),
        "BERTScore_F1": round(sum(all_bs_f1) / len(all_bs_f1), 4),
        "Cosine_Similarity": round(sum(max_cosines) / len(max_cosines), 4),
    }

In [ ]:
def build_dual_indexes(df, model, tokenizer):
    # vectors storage
    keys_to_idiom = []
    keys_to_literal = []

    vals_idiomatic = df['Idiomatic_Sent'].tolist()
    vals_literal = df['Literal_Sent'].tolist()

    device = next(model.parameters()).device

    model.eval()
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Building Indexes"):
        with torch.no_grad():
            inputs_lit = tokenizer(row['Literal_Sent'], return_tensors="pt", truncation=True).to(device)
            out_lit = model(**inputs_lit, output_hidden_states=True)
            vec_lit = out_lit.hidden_states[-1][0, -1, :].to(torch.float32).cpu().numpy()
            keys_to_idiom.append(vec_lit)

            inputs_idm = tokenizer(row['Idiomatic_Sent'], return_tensors="pt", truncation=True).to(device)
            out_idm = model(**inputs_idm, output_hidden_states=True)
            vec_idm = out_idm.hidden_states[-1][0, -1, :].to(torch.float32).cpu().numpy()
            keys_to_literal.append(vec_idm)

    # FAISS indexes
    index_to_idiom = faiss.IndexFlatL2(len(keys_to_idiom[0]))
    index_to_idiom.add(np.array(keys_to_idiom).astype('float32'))

    index_to_literal = faiss.IndexFlatL2(len(keys_to_literal[0]))
    index_to_literal.add(np.array(keys_to_literal).astype('float32'))

    return (index_to_idiom, vals_idiomatic), (index_to_literal, vals_literal)

In [ ]:
def knn_generate_step(input_text, index_data, model, tokenizer, lambda_knn=0.4):
    index, target_sentences = index_data
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        # model's probabilities
        p_lm = torch.softmax(outputs.logits[0, -1, :], dim=-1)
        query_vec = outputs.hidden_states[-1][0, -1, :].to(torch.float32).cpu().numpy().reshape(1, -1)

    distances, indices = index.search(query_vec, k=8)

    # knn's probabilities
    p_knn = torch.zeros_like(p_lm)
    for idx in indices[0]:
        target_tokens = tokenizer.encode(target_sentences[idx], add_special_tokens=False)
        unique_tokens = set(target_tokens)
        for t in unique_tokens:
            if t < tokenizer.vocab_size:
                p_knn[t] += 1.0 / len(indices[0])

    # reset p_knn for tokens whose p_lm is too low
    mask = (p_lm > 0.001).float()
    p_knn = p_knn * mask

    # p_knn normalization
    if p_knn.sum() > 0:
        p_knn = p_knn / p_knn.sum()

    final_probs = (1 - lambda_knn) * p_lm + lambda_knn * p_knn  # interpolation
    next_token = torch.argmax(final_probs, dim=-1).unsqueeze(0) # greedy search

    return tokenizer.decode(next_token)

In [ ]:
def knn_generate_full_sentence(prompt, index_data, model, tokenizer, max_len=50):
    current_text = prompt
    # so that model doesn't finish the sentence too early
    min_tokens = 5

    for i in range(max_len):
        next_token = knn_generate_step(current_text, index_data, model, tokenizer)

        if tokenizer.eos_token in next_token:
            break

        current_text += next_token

    result = current_text.replace(prompt, "").strip()

    return result

In [ ]:
clean_model_name = model_name.split('/')[-1]
model_dir = os.path.join("knn-lm", clean_model_name)
os.makedirs(model_dir, exist_ok=True)

In [ ]:
# building indexes
idx_data_idiom, idx_data_lit = build_dual_indexes(test_df, model, tokenizer)

In [ ]:
ref_to_idiom = test_df.groupby('Literal_Sent')['Idiomatic_Sent'].apply(list).to_dict()
ref_to_literal = test_df.groupby('Idiomatic_Sent')['Literal_Sent'].apply(list).to_dict()

ENG

In [ ]:
directions = ['to_idiomatic', 'to_literal']

for direction in directions:
    if direction == 'to_idiomatic':
        index_data = idx_data_idiom
        input_col, ref_col = 'Literal_Sent', 'Idiomatic_Sent'
        prompt_template = "Rewrite the sentence to make it idiomatic.\nSentence: {input_text}\nOutput:"
        lookup = ref_to_idiom
    else:
        index_data = idx_data_lit
        input_col, ref_col = 'Idiomatic_Sent', 'Literal_Sent'
        prompt_template = "Replace the idiom in the sentence with a literal expression.\nSentence: {input_text}\nOutput:"
        lookup = ref_to_literal

    results = []
    for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Generating {direction}"):
        input_text = row[input_col]
        prompt = prompt_template.format(input_text=input_text)
        gen_text = knn_generate_full_sentence(prompt, index_data, model, tokenizer)

        results.append({
            'idiom': row['Idiom'],
            "input_text": input_text,
            "reference": row[ref_col],
            "generated_text": gen_text
        })

    csv_path = os.path.join(model_dir, f"knn_results_{direction}.csv")
    pd.DataFrame(results).to_csv(csv_path, index=False)

    print(f"Calculating metrics for {direction}...")
    metrics = calculate_metrics(csv_path, lookup)

    metrics["direction"] = direction
    stats_path = os.path.join(model_dir, "knn_statistics.csv")

    stats_df = pd.DataFrame([metrics])
    if not os.path.exists(stats_path):
        stats_df.to_csv(stats_path, index=False)
    else:
        stats_df.to_csv(stats_path, mode='a', header=False, index=False)

RU

In [ ]:
directions = ['to_idiomatic', 'to_literal']

for direction in directions:

    if direction == 'to_idiomatic':
        index_data = idx_data_idiom
        input_col, ref_col = 'Literal_Sent', 'Idiomatic_Sent'
        prompt_template = "Перепиши предложение, добавив идиому.\nSentence: {input_text}\nВывод:"
        lookup = ref_to_idiom
    else:
        index_data = idx_data_lit
        input_col, ref_col = 'Idiomatic_Sent', 'Literal_Sent'
        prompt_template = "Замени идиому в предложении буквальным выражением.\nSentence: {input_text}\nВывод:"
        lookup = ref_to_literal

    results = []
    for i, row in tqdm(train_df.iterrows(), total=len(train_df), desc=f"Generating {direction}"):
        input_text = row[input_col]
        prompt = prompt_template.format(input_text=input_text)
        gen_text = knn_generate_full_sentence(prompt, index_data, model, tokenizer)

        results.append({
            'idiom': row['Idiom'],
            "input_text": input_text,
            "reference": row[ref_col],
            "generated_text": gen_text
        })

    csv_path = os.path.join(model_dir, f"knn_results_{direction}.csv")
    pd.DataFrame(results).to_csv(csv_path, index=False)

    print(f"Calculating metrics for {direction}...")
    metrics = calculate_metrics(csv_path, lookup)

    metrics["direction"] = direction
    stats_path = os.path.join(model_dir, "knn_statistics.csv")

    stats_df = pd.DataFrame([metrics])
    if not os.path.exists(stats_path):
        stats_df.to_csv(stats_path, index=False)
    else:
        stats_df.to_csv(stats_path, mode='a', header=False, index=False)